# Introdução às Redes Neurais e Deep Learning

## LSTM (Long Short-Term Memory)

A rede do tipo Long Short-term Memory (LSTM)é uma modificação da rede neural recorrente (RNN) padrão destinada a mitigar o problema do desaparecimento do gradiente comumente encontrado pelas RNNs tradicionais. Seu objetivo é fornecer uma memória de curto prazo para RNN que possa durar milhares de passos de tempo (daí o nome "Longa memória de curto prazo").

A rede LSTM acrescenta ao estado oculto $h_t$ das RNNs um novo estado, o estado da célula $C_t$, cuja função é emular uma memória interna de longo prazo carregando informação através do tempo. Além disso são usadas três portas nesta arquitetura:
* Porta de esquecimento $f_t$, que decide o que deve ser descartado do estado da célula no  instante anterior. Valores próximos de 0 são "esquecidos", e valores próximos de 1 "mantidos".
* Porta de entrada: combina duas funções,  $i_t$ decide o montante da nova informação que entra na célula e $\tilde{C}_t$ decide o que deve ser atualizado ao estado da célula.
* Porta de saída $o_t$ decide o quanto do estado atual da célula sera exposto como estado oculto $h_t$.
<div>
<img src="Files/ChatGPT Image LSTM célula.png" width="700" height="600">   
</div>  
No diagrama acima $\sigma$ é a função de ativação sigmoide, $\tanh$ é a tangente hiperbólica, $\otimes$ e $\oplus$ são o produto e adição elemento por elemento. As equações que governam o fluxo de informações nas portas e estados são:
$$ \begin{aligned}
    f_t &= \sigma(W_fz_t+b_f)\\
    i_t &= \sigma(W_iz_t+b_i)\\
    \tilde C_t &= \tanh(W_cz_t+b_c)\\
    o_t &= \sigma(W_oz_t+b_o)\\
    C_t &= f_t\otimes C_{t-1}\oplus i_t\otimes\tilde C_t\\
    h_t &= o_t\otimes\tanh(C_t)
    \end{aligned}$$ 
com $z_t=[h_{t-1},x_t]$.

Um dos principais mecanismos é o controle da atualização do estado da célula $C_t = f_t\otimes C_{t-1}\oplus i_t\otimes\tilde C_t$. Ao longo deste caminho a derivada local é
$$ \frac{\partial C_t}{\partial C_{t-1}}=f_t,$$
Assim, quando $f_t$ permanece próximo de 1 o gradiente pode ser preservado por longos períodos de tempo, ajudando a mitigar o problema de gradientes evanescentes. 

Uma última porta final de saída (não mostrada acima) é definida como 
$$\hat{y}_t=w_y \cdot h_t +b_y.$$

### Retropropagação pelo tempo (BPTT)

 A retropropagação inicia, como sempre, pela derivada da função de perdas da saída da célula em respeito à própria saída. A derivada depende obviamente da função *Loss* escolhida.  Temos então. inicialmente, $\frac{\partial L}{\partial \hat{y}}$.
 A derivada com respeito aos pesos são obtidas através do método das derivadas parciais (veja as equações de propagação direta na figura acima):

  $$\frac{\partial L}{\partial W_y}= \sum_t \frac{\partial L}{\partial \hat{y}_t} \otimes \frac{\partial \hat{y}_t}{\partial W_y}=\sum_t\frac{\partial L}{\partial \hat{y}_t} \cdot h_t^T$$
  $$\frac{\partial L}{\partial b_y}=\sum_t\frac{\partial L}{\partial \hat{y_t}},$$

Agora, para os outros gradientes, precisaremos primeiro determinar os gradientes em relação ao estado oculto $h_t$ e ao estado da célula $C_t$. Entretanto, é importante notar que em uma LSTM , $h_t$ e $C_t$ influenciam o futuro por múltiplos caminhos simultaneamente. Com efeito, $h_t=o_t \otimes \tanh(C_t)$ e ao mesmo é utilizado em  $h_{t+1}$. Portanto, podemos escrever que

 $$\frac{dL}{dh_t}=\frac{\partial L_t}{\partial h_t}+\left.\frac{dL}{dh_t}\right|_{\text{futuro}}$$
  onde a primeira derivada no lado direito é o gradiente da função *Loss* em relação ao estado presente e a segunda é o gradiente  da função *Loss* no instante $t+1$ em relação à ${h_t}$. Assim,
  $$\frac{\partial L}{\partial h_t}=\frac{\partial L}{\partial \hat{y}_t}\cdot W_y+\left.\frac{\partial L}{\partial h_t}\right|_{\text{futuro}}$$

Seja $z_t$ a concatenação do estado oculto na saída do passo anterior e da entrada no passo atual, como mostrado nas expressões da figura acima, ou seja, $z_t=\begin{bmatrix}h_{t-1} \\ x_t\end{bmatrix}$. $z_t$ está presente nas definições de $f_t,\;i_t,\;\tilde{c}_t\;\text{e}\;o_t$. Daí que 
$$\frac{\partial L}{\partial z_t}=W_f^T\frac{\partial L}{\partial a_{f,t}}+W_i^T\frac{\partial L}{\partial a_{i,t}}+W_c^T\frac{\partial L}{\partial a_{c,t}}+W_o^T\frac{\partial L}{\partial a_{o,t}}=\begin{bmatrix}\left.\frac{\partial L}{\partial h_{t-1}}\right|_{\text{via}z_t}\\ \frac{\partial L}{\partial x_t}\end{bmatrix}$$
onde $a_{f,t},a_{i,t},a_{c,t},a_{o,t}$ são as entradas das respectivas funções de ativação.Por exemplo,
$$\frac{\partial L}{\partial a_{f,t}}=\frac{\partial L}{\partial f_t}\otimes f_t\otimes(1-f_t).$$

Para o estado da célula, $C_t$, temos também que $C_{t+1}=f_{t+1}C_t+ i_{t+1}\tilde{C}_{t+1}$, de forma que além do estado oculto o estado da célula no instante $t+1$ também de pende do estado no instante $t$, e daí
$$\frac{\partial L}{\partial C_t}=\frac{\partial L}{\partial h_t}\otimes\frac{\partial h_t}{\partial C_t}+\left.\frac{\partial L}{\partial C_t}\right|_{\text{futuro}} ,$$
onde $\left.\frac{\partial L}{\partial C_t}\right|_{\text{futuro}}=f_{t+1}\frac{\partial L}{\partial C_{t+1}}.$  Como $h_t=o_t\otimes\tanh(C_t)$, temos:
$$\frac{\partial h_t}{\partial C_t}=o_t\otimes [1-\tanh^2(C_t)].$$
Portanto 
$$\frac{\partial L}{\partial C_t}=\frac{\partial L}{\partial h_t}\otimes o_t\otimes [1-\tanh^2(C_t)] +f_{t+1}\otimes \frac{\partial L}{\partial C_{t+1}}$$

Os outros gradientes podem ser agora obtidos com facilidade:


$$\frac{\partial L}{\partial W_o}=\sum_t\left(\frac{\partial L}{\partial h_t}\otimes\frac{\partial h_t}{\partial o_t}\otimes\frac{\partial o_t}{\partial W_o}\right)=\sum_t\left[\frac{\partial L}{\partial h_t}\otimes \tanh(C_t)\right]\otimes o_t\otimes(1-o_t)\cdot z_t^T$$

$$\frac{\partial L}{\partial b_o}=\sum_t\left(\frac{\partial L}{\partial h_t}\otimes\frac{\partial h_t}{\partial o_t}\otimes\frac{\partial o_t}{\partial b_o}\right)=\sum_t\left[\frac{\partial L}{\partial h_t}\otimes \tanh(C_t)\right]\otimes o_t\otimes(1-o_t)$$

$$\frac{\partial L}{\partial W_f}=\sum_t\left(\frac{\partial L}{\partial C_t}\otimes\frac{\partial C_t}{\partial f_t}\otimes\frac{\partial f_t}{\partial W_f}\right)=\sum_t\left[\frac{\partial L}{\partial C_t}\otimes C_{t-1}\right]\otimes f_t\otimes(1-f_t)\cdot z_t^T$$

$$\frac{\partial L}{\partial b_f}=\sum_t\left(\frac{\partial L}{\partial C_t}\otimes\frac{\partial C_t}{\partial f_t}\otimes\frac{\partial f_t}{\partial b_f}\right)=\sum_t\left[\frac{\partial L}{\partial C_t}\otimes C_{t-1}\right]\otimes f_t\otimes(1-f_t)$$

$$\frac{\partial L}{\partial W_i}=\sum_t\left(\frac{\partial L}{\partial C_t}\otimes\frac{\partial C_t}{\partial i_t}\otimes\frac{\partial i_t}{\partial W_i}\right)=\sum_t\left[\frac{\partial L}{\partial C_t}\otimes \tilde{C_t}\right]\otimes i_t\otimes(1-i_t)\cdot z_t^T$$

$$\frac{\partial L}{\partial b_i}=\sum_t\left(\frac{\partial L}{\partial C_t}\otimes\frac{\partial C_t}{\partial i_t}\otimes\frac{\partial i_t}{\partial b_i}\right)=\sum_t\left[\frac{\partial L}{\partial C_t}\otimes \tilde{C_t}\right]\otimes i_t\otimes(1-i_t)$$

$$\frac{\partial L}{\partial w_c}=\sum_t\left(\frac{\partial L}{\partial C_t}\otimes\frac{\partial C_t}{\partial \tilde{C}_t}\otimes\frac{\partial \tilde{C}_t}{\partial w_c}\right)=\sum_t\left[\frac{\partial L}{\partial C_t}\otimes i_t\right]\otimes (1-\tilde{C}_t^2)\cdot z_t^T$$

$$\frac{\partial L}{\partial b_c}=\sum_t\left(\frac{\partial L}{\partial C_t}\otimes\frac{\partial C_t}{\partial \tilde{C}_t}\otimes\frac{\partial \tilde{C_t}}{\partial b_c}\right)=\sum_t\left[\frac{\partial L}{\partial C_t}\otimes i_t\right]\otimes (1-\tilde{C}_t^2)$$

In [1]:
import numpy as np

class LSTM:
    def __init__(self, lr=0.1):

        self.lr = lr

        # pesos
        self.Wf = np.array([[0.7, 0.2]])
        self.Wi = np.array([[0.5, 0.1]])
        self.Wo = np.array([[0.9, 0.3]])
        self.Wc = np.array([[0.6, 0.2]])

        self.bf = np.array([[0.1]])
        self.bi = np.array([[0.1]])
        self.bo = np.array([[0.1]])
        self.bc = np.array([[0.1]])

        self.params = ["Wf", "Wi", "Wo", "Wc",
                       "bf", "bi", "bo", "bc"]

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    # ======================================
    # FORWARD
    # ======================================

    def forward(self, X):

        h = np.zeros((1, 1))
        c = np.zeros((1, 1))

        self.cache = []

        for x in X:

            x = np.array([[x]])

            # Concatenação [x_t, h_(t-1)]
            z = np.vstack((x, h))

            f = self.sigmoid(self.Wf @ z + self.bf)
            i = self.sigmoid(self.Wi @ z + self.bi)
            o = self.sigmoid(self.Wo @ z + self.bo)

            c_til = np.tanh(self.Wc @ z + self.bc)

            c_new = f * c + i * c_til
            h_new = o * np.tanh(c_new)

            # Armazenar valores para o BPTT
            self.cache.append({
                "z": z,
                "f": f,
                "i": i,
                "o": o,
                "c_til": c_til,
                "c_prev": c,
                "c": c_new,
                "h": h_new
            })

            h = h_new
            c = c_new

        return h

    # ======================================
    # BACKWARD THROUGH TIME
    # ======================================

    def backward(self, target):

        # Inicialização dos gradientes
        self.grads = {
            p: np.zeros_like(getattr(self, p))
            for p in self.params
        }

        h_final = self.cache[-1]["h"]

        # Loss = (h_final - target)^2
        loss = (h_final - target)**2

        # Derivada da perda no último instante
        dh_next = 2 * (h_final - target)

        # Não há perda futura após o último instante
        dc_next = np.zeros((1, 1))

        # Percorrer os instantes em ordem inversa
        for t in reversed(range(len(self.cache))):

            dados = self.cache[t]

            z = dados["z"]
            f = dados["f"]
            i = dados["i"]
            o = dados["o"]
            c_til = dados["c_til"]
            c_prev = dados["c_prev"]
            c = dados["c"]

            # Gradiente da porta de saída
            do = dh_next * np.tanh(c)
            delta_o = do * o * (1 - o)

            # Gradiente total do estado da célula
            dc = (
                dh_next * o * (1 - np.tanh(c)**2)
                + dc_next
            )

            # Porta de esquecimento
            df = dc * c_prev
            delta_f = df * f * (1 - f)

            # Porta de entrada
            di = dc * c_til
            delta_i = di * i * (1 - i)

            # Estado candidato
            dc_til = dc * i
            delta_c = dc_til * (1 - c_til**2)

            # Gradientes dos pesos
            self.grads["Wf"] += delta_f @ z.T
            self.grads["Wi"] += delta_i @ z.T
            self.grads["Wo"] += delta_o @ z.T
            self.grads["Wc"] += delta_c @ z.T

            # Gradientes dos bias
            self.grads["bf"] += delta_f
            self.grads["bi"] += delta_i
            self.grads["bo"] += delta_o
            self.grads["bc"] += delta_c

            # Gradiente em relação à concatenação z
            dz = (
                self.Wf.T @ delta_f
                + self.Wi.T @ delta_i
                + self.Wo.T @ delta_o
                + self.Wc.T @ delta_c
            )

            # z = [x_t, h_(t-1)]
            # Segunda componente: gradiente futuro de h
            dh_next = dz[1:2, :]

            # Caminho direto da memória
            dc_next = dc * f

        return loss.item()

    # ======================================
    # ATUALIZAÇÃO DOS PESOS
    # ======================================

    def update(self):

        for p in self.params:
            valor = getattr(self, p)
            setattr(
                self, p,
                valor - self.lr * self.grads[p]
            )

    # ======================================
    # EXIBIÇÃO DOS PARÂMETROS
    # ======================================

    def show_parameters(self):

        for p in self.params:
            print(f"{p} = {getattr(self, p)}")


In [2]:

# Dados de entrada
X = [0.5, 0.8]
target = 0.4

# Criar a LSTM
model = LSTM(lr=0.1)

# Forward
h2 = model.forward(X)

print("======== FORWARD ========")

for t, dados in enumerate(model.cache, start=1):
    print(f"\nTempo {t}")
    for nome in ["f", "i", "o", "c_til", "c", "h"]:
        print(f"{nome} = {dados[nome].item():.8f}")

# Backward
loss = model.backward(target)

print("\n======== LOSS ========")
print("Loss =", loss)

print("\n======== GRADIENTES ========")

for p, grad in model.grads.items():
    print(f"{p} = {grad}")

# Atualização
model.update()

print("\n======== PESOS ATUALIZADOS ========")
model.show_parameters()


======== FORWARD ========

Tempo 1
f = 0.61063923
i = 0.58661758
o = 0.63413559
c_til = 0.37994896
c = 0.22288474
h = 0.13904427

Tempo 2
f = 0.66547932
i = 0.62572132
o = 0.70301848
c_til = 0.54258286
c = 0.48783084
h = 0.31811102

======== LOSS ========
Loss = 0.006705805491948991

======== GRADIENTES ========
Wf = [[-0.00363458 -0.00063171]]
Wi = [[-0.01252672 -0.00161779]]
Wo = [[-0.01275452 -0.00215138]]
Wc = [[-0.04987575 -0.00562109]]
bf = [[-0.00454322]]
bi = [[-0.01807241]]
bo = [[-0.01622547]]
bc = [[-0.07549551]]

======== PESOS ATUALIZADOS ========
Wf = [[0.70036346 0.20006317]]
Wi = [[0.50125267 0.10016178]]
Wo = [[0.90127545 0.30021514]]
Wc = [[0.60498758 0.20056211]]
bf = [[0.10045432]]
bi = [[0.10180724]]
bo = [[0.10162255]]
bc = [[0.10754955]]
